In [1]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [3]:
# Create a simple one layer model using a linear layer
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleLinearModel, self).__init__()
        # Define a single linear layer
        self.linear = nn.Linear(input_size, 10)
        self.linear2 = nn.Linear(10, 20)
        self.linear3 = nn.Linear(20, 15)
        self.linear4 = nn.Linear(15, output_size)

    def forward(self, x):
        # Pass input through the linear layer
        output = self.linear(x)
        output = self.linear2(output)
        output = self.linear3(output)
        output = self.linear4(output)
        return output


# Example usage
model = SimpleLinearModel(input_size=10, output_size=5)
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleLinearModel(
  (linear): Linear(in_features=10, out_features=10, bias=True)
  (linear2): Linear(in_features=10, out_features=20, bias=True)
  (linear3): Linear(in_features=20, out_features=15, bias=True)
  (linear4): Linear(in_features=15, out_features=5, bias=True)
)
Model weights:
linear.weight: torch.Size([10, 10])
  Weight values (first 5): tensor([ 0.2616,  0.0804,  0.2223, -0.1296, -0.1498], grad_fn=<SliceBackward0>)
linear.bias: torch.Size([10])
  Bias values (first 5): tensor([0.2291, 0.3157, 0.2969, 0.0681, 0.1829], grad_fn=<SliceBackward0>)
linear2.weight: torch.Size([20, 10])
  Weight values (first 5): tensor([-0.1765,  0.0547,  0.1009,  0.0140, -0.2883], grad_fn=<SliceBackward0>)
linear2.bias: torch.Size([20])
  Bias values (first 5): tensor([-0.2461, -0.1137, -0.0481,  0.2590,  0.0162], grad_fn=<SliceBackward0>)
linear3.weight: torch.Size([15, 20])
  Weight values (first 5): tensor([ 0.1015,  0.0875, -0.1367, -0.0717, -0.1742], grad_fn=<SliceBackward0>)
linear3.bias:

In [4]:
# export to onnx
onnx.export(model, torch.randn(1, 10), "simple_model.onnx", export_params=True, opset_version=11)

In [5]:
# run the model with pytorch
input_data = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]])
with torch.no_grad():
    output = model(input_data)

print(output)

tensor([[-0.0426, -0.3117,  0.1413,  0.2507,  0.2906]])
